# 02. Decorators & Nested Traces

In this notebook, you will learn how to use the `@observe()` decorator to trace complex pipelines in detail.

## What you will learn
- Automatic function tracing with the `@observe()` decorator
- Creating hierarchical traces from nested function calls
- Adding metadata to traces with `propagate_attributes()` (user_id, session_id, tags)
- Simulating a real RAG (Retrieval-Augmented Generation) pipeline

## What is @observe()?

When you add `@observe()` to a function:
- Function inputs/outputs are automatically captured
- Execution time is measured
- Errors are automatically recorded
- Nested `@observe()` functions are displayed in a parent-child hierarchy

## Step 1: Initial Setup

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()
os.environ["ANTHROPIC_API_KEY"] = os.environ["CLAUDE_API_KEY"]

from langfuse import get_client, observe, propagate_attributes
from opentelemetry.instrumentation.anthropic import AnthropicInstrumentor
from anthropic import Anthropic

AnthropicInstrumentor().instrument()
langfuse = get_client()
client = Anthropic()

print("✅ Setup complete")

## Step 2: Basic @observe() Usage

Simply add `@observe()` above your function.

In [ ]:
@observe()
def summarize_text(text: str, language: str = "Korean") -> str:
    """Function that summarizes text — automatically traced"""
    response = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=256,
        messages=[
            {
                "role": "user",
                "content": f"Please summarize the following text in 2 sentences in {language}:\n\n{text}"
            }
        ]
    )
    return response.content[0].text


sample_text = """
Langfuse is an open-source LLM engineering platform that helps teams collaboratively debug, 
analyze, and iterate on their LLM applications. It provides observability features including 
tracing of LLM calls, prompt management, evaluation tools, and dataset management. 
Langfuse integrates with popular frameworks like LangChain, LlamaIndex, and provides 
native SDKs for Python and JavaScript.
"""

result = summarize_text(sample_text)
print(result)

langfuse.flush()

> **Check in the dashboard:**
> Find the `summarize_text` trace in the Traces list.
> You will see that the function arguments (`text`, `language`) and the return value have been automatically captured.

## Step 3: Nested Traces

When an `@observe()` function calls another `@observe()` function,
Langfuse visualizes them in a hierarchical structure.

```
rag_pipeline  (Top-level Trace)
  ├─ retrieve_context  (Span)
  ├─ build_prompt      (Span)
  └─ generate_answer   (Span)
       └─ [Claude API call]  (Generation)
```

In [ ]:
# Simple document database (simulation)
DOCUMENT_DB = {
    "langfuse": [
        "Langfuse is an open-source observability platform for LLM applications.",
        "Langfuse provides tracing, prompt management, and evaluation features.",
        "The Langfuse Python SDK is based on OpenTelemetry.",
    ],
    "claude": [
        "Claude is a large language model developed by Anthropic.",
        "Claude is designed with goals of safety, helpfulness, and honesty.",
        "Claude 3.5 Sonnet has excellent reasoning capabilities.",
    ],
    "rag": [
        "RAG (Retrieval-Augmented Generation) is a technique for injecting external knowledge into LLMs.",
        "RAG is effective at reducing LLM hallucinations.",
        "A RAG pipeline consists of three stages: retrieval, augmentation, and generation.",
    ]
}


@observe()  # Span: Document retrieval
def retrieve_context(query: str) -> list[str]:
    """Retrieve relevant documents by query keywords"""
    query_lower = query.lower()
    
    for keyword, docs in DOCUMENT_DB.items():
        if keyword in query_lower:
            return docs
    
    # If no match, return all documents
    all_docs = []
    for docs in DOCUMENT_DB.values():
        all_docs.extend(docs)
    return all_docs[:3]


@observe()  # Span: Prompt construction
def build_prompt(query: str, context: list[str]) -> str:
    """Build a prompt that includes the retrieved context"""
    context_str = "\n".join(f"- {doc}" for doc in context)
    return f"""Answer the question based on the following context.

Context:
{context_str}

Question: {query}

If the answer is not found in the context, say you don't know."""


@observe()  # Span: LLM generation (Generation is nested inside)
def generate_answer(prompt: str) -> str:
    """Generate an answer using Claude"""
    response = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=512,
        messages=[{"role": "user", "content": prompt}]
    )
    return response.content[0].text


@observe()  # Top-level Trace: Full pipeline
def rag_pipeline(query: str) -> str:
    """Execute the full RAG pipeline"""
    context = retrieve_context(query)
    prompt = build_prompt(query, context)
    answer = generate_answer(prompt)
    return answer


query = "What is Langfuse?"
answer = rag_pipeline(query)

print(f"Question: {query}")
print(f"\nAnswer: {answer}")

langfuse.flush()

> **Check in the dashboard:**
> Click on the `rag_pipeline` trace to see the nested hierarchy:
> `retrieve_context` → `build_prompt` → `generate_answer` → `[Claude call]`

## Step 4: Adding Metadata with propagate_attributes()

In a real service, you need to track which user made a request and in which session.
`propagate_attributes()` lets you assign common attributes to all Spans within a specific execution scope.

In [ ]:
import uuid


@observe()
def handle_user_request(user_id: str, session_id: str, query: str) -> str:
    """Entry point for handling user requests"""
    with propagate_attributes(
        user_id=user_id,
        session_id=session_id,
        tags=["rag", "production"],
        metadata={"query_length": len(query), "version": "1.0"}
    ):
        return rag_pipeline(query)


result = handle_user_request(
    user_id="user_42",
    session_id=str(uuid.uuid4()),
    query="What is RAG?"
)

print(result)
langfuse.flush()

> **Check in the dashboard:**
> Click on the trace to see `user_id`, `session_id`, and `tags` displayed at the top.
> You can also filter all traces for a specific user under the **Users** tab.

## Step 5: Error Tracing

`@observe()` automatically captures exceptions even when errors occur.

In [ ]:
@observe()
def process_with_validation(query: str) -> str:
    if not query or len(query.strip()) == 0:
        raise ValueError("Query is empty")
    
    if len(query) > 1000:
        raise ValueError(f"Query is too long: {len(query)} characters (max 1000)")
    
    return rag_pipeline(query)


# Normal case
try:
    result = process_with_validation("What kind of model is Claude?")
    print("[Success]")
    print(result)
except Exception as e:
    print(f"[Error] {e}")

langfuse.flush()

# Error case
try:
    result = process_with_validation("")
except ValueError as e:
    print(f"\n[Expected error] {e}")

langfuse.flush()

## Summary

| Concept | Description |
|------|------|
| `@observe()` | Decorator that automatically traces a function |
| Nested `@observe()` | Reflects the call hierarchy directly as a trace hierarchy |
| `propagate_attributes()` | Assigns common attributes to all Spans within an execution scope |
| Error tracing | Automatically captures error information when exceptions occur |

Next: **03_prompt_management.ipynb** → Managing prompts outside of code